# 📚 Study Assistant — 실험 노트북

**파이프라인 구조**
```
PDF  →  [1] OCR (Mistral)  →  [2] Chunking (GPT)  →  [3] Pipeline (요약+퀴즈)  →  Obsidian Vault
                                                   ↘  [4] Quiz Generator
                                                   ↘  [5] Graph RAG
```

**각 셀은 독립 실행 가능합니다.** 이전 단계 결과가 Drive에 저장되어 있으면 해당 단계부터 시작할 수 있습니다.

---
## ⚙️ 0. 환경 설정

In [ ]:
# Google Drive 마운트
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 패키지 설치 (GitHub에서 직접 설치)
!pip install -q git+https://github.com/syuririk/SA.git

# 의존성 확인
!pip install -q mistralai openai pyyaml nest_asyncio pyvis pypdf

In [ ]:
import os

# ── API 키 설정 ──────────────────────────────────────────
# 방법 A: 직접 입력 (간단하지만 노트북에 키가 노출됨)
# os.environ["OPENAI_API_KEY"]  = "sk-..."
# os.environ["MISTRAL_API_KEY"] = "..."

# 방법 B: Colab Secrets 사용 (권장) — 왼쪽 사이드바 🔑 아이콘
from google.colab import userdata
os.environ["OPENAI_API_KEY"]  = userdata.get("OPENAI_API_KEY")
os.environ["MISTRAL_API_KEY"] = userdata.get("MISTRAL_API_KEY")

print("✅ OpenAI key:",  "설정됨" if os.environ.get("OPENAI_API_KEY")  else "❌ 미설정")
print("✅ Mistral key:", "설정됨" if os.environ.get("MISTRAL_API_KEY") else "❌ 미설정")

In [ ]:
from study_assistant import Config, print_all

# ── Config 초기화 ────────────────────────────────────────
# 기본값 사용 (config.yaml이 없어도 동작)
cfg = Config()

# Drive 경로 커스터마이즈가 필요하면 아래 주석 해제
# cfg.set("paths.root", "/content/drive/MyDrive/나의폴더")

# Mistral API 키를 config에 명시적으로 전달 (환경변수 대신)
# cfg.set("api_keys.mistral", "your-key-here")

cfg.show()
cfg.ensure_dirs()

# 현재 데이터 현황 출력
print_all(cfg)

---
## 📄 1단계: OCR

PDF → 페이지별 `.md` 파일로 변환 (Mistral OCR API 사용)  
결과는 `OCR/{교재명}/page_NNNN.md` 형태로 저장됩니다.

In [ ]:
from study_assistant import list_pdfs, print_pdfs

pdfs = list_pdfs(cfg.pdf_dir)
print_pdfs(pdfs)

# 처리할 교재 선택 (인덱스 또는 파일명으로)
BOOK_INDEX = cfg.book_index   # config.yaml의 book.index 값
if pdfs:
    print(f"\n선택된 교재: [{BOOK_INDEX}] {pdfs[BOOK_INDEX]['name']}")

In [ ]:
from study_assistant import run_ocr, list_pdfs

pdfs = list_pdfs(cfg.pdf_dir)
if not pdfs:
    raise FileNotFoundError(f"PDF 없음: {cfg.pdf_dir} 폴더에 PDF를 넣어주세요.")

selected = pdfs[BOOK_INDEX]
BOOK_NAME = selected["name"]
PDF_PATH  = str(selected["path"])

print(f"📖 교재: {BOOK_NAME}")
print(f"📂 경로: {PDF_PATH}")
print(f"📦 크기: {selected['size_mb']} MB")

ocr_out = run_ocr(cfg, PDF_PATH, BOOK_NAME)
print(f"\n✅ OCR 완료 → {ocr_out}")

In [ ]:
# OCR 결과 샘플 확인
from pathlib import Path

ocr_dir = cfg.book_ocr_dir(BOOK_NAME)
pages = sorted(ocr_dir.glob("page_*.md"))
print(f"총 {len(pages)}페이지 저장됨")
print("\n--- 첫 페이지 미리보기 ---")
if pages:
    print(pages[0].read_text(encoding="utf-8")[:800])

---
## ✂️ 2단계: 청킹

페이지들을 논리적 섹션으로 그룹화합니다.  
- `interactive=True` (기본): 편집 UI 표시  
- `interactive=False`: LLM 자동 분류 후 즉시 저장 (자동화용)

In [ ]:
from study_assistant import run_chunking

# BOOK_NAME이 앞 셀에서 설정되지 않은 경우 직접 지정
# BOOK_NAME = "교재명"

# interactive=False: 자동화 모드 (캐시 있으면 재사용, 없으면 LLM 청킹 후 저장)
chunks = run_chunking(cfg, BOOK_NAME, interactive=False)
print(f"\n✅ 청크 수: {len(chunks)}개")

In [ ]:
# interactive=True: 편집 UI (수동으로 청크 조정하고 싶을 때)
# chunks = run_chunking(cfg, BOOK_NAME, interactive=True)

In [ ]:
# 청크 분포 확인
from collections import Counter

type_counts = Counter(c.get("type", "?") for c in chunks)
print("청크 타입 분포:")
for t, n in type_counts.items():
    print(f"  {t}: {n}개")

print("\n청크 목록:")
for i, c in enumerate(chunks):
    pp = c["pages"]
    ps = f"p{pp[0]}-{pp[-1]}" if len(pp) > 1 else f"p{pp[0]}"
    print(f"  [{i:2d}] [{c.get('type','?'):7s}] {ps:12s} ({len(pp):2d}p) | {c['title']}")

---
## 🏭 3단계: 파이프라인 (요약 + 퀴즈 → Vault)

청크별로 요약과 퀴즈를 생성하여 Obsidian Vault에 저장합니다.  
- 기존 Vault가 있으면 덮어쓰기 여부를 물어봅니다 (`overwrite=True`로 강제 가능)

In [ ]:
from study_assistant import run_pipeline

# overwrite=False (기본): 기존 Vault 존재 시 덮어쓸지 물어봄
# overwrite=True: 묻지 않고 덮어씀
results, vault_dir = run_pipeline(cfg, BOOK_NAME, overwrite=False)

print(f"\n✅ Vault 생성 완료: {vault_dir}")
print(f"   총 {len(results)}개 파일")

In [ ]:
# Vault 구조 확인
from pathlib import Path

vault = cfg.book_vault_dir(BOOK_NAME)
for folder in sorted(vault.iterdir()):
    if folder.is_dir():
        files = list(folder.glob("*.md"))
        print(f"  📁 {folder.name}/  ({len(files)}개)")
        for f in files[:3]:
            print(f"       {f.name}")
        if len(files) > 3:
            print(f"       ... ({len(files)-3}개 더)")
    elif folder.suffix == ".md":
        print(f"  📄 {folder.name}")

In [ ]:
# 생성된 요약 파일 미리보기
summaries = list((vault / "Summaries").glob("*.md"))
if summaries:
    print(f"요약 파일 {len(summaries)}개 중 첫 번째:")
    print("="*60)
    print(summaries[0].read_text(encoding="utf-8")[:1200])
else:
    print("요약 파일 없음")

In [ ]:
# 생성된 퀴즈 파일 미리보기
quizzes = list((vault / "Quizzes").glob("*.md"))
if quizzes:
    print(f"퀴즈 파일 {len(quizzes)}개 중 첫 번째:")
    print("="*60)
    print(quizzes[0].read_text(encoding="utf-8")[:1200])
else:
    print("퀴즈 파일 없음")

---
## ❓ 4단계: Quiz Generator

Vault의 요약/퀴즈 파일을 바탕으로 새로운 퀴즈를 생성합니다.  
지원 타입: `multiple_choice`, `ox`, `short_answer`, `fill_in_blank`

In [ ]:
from study_assistant import generate_quiz

result = generate_quiz(
    cfg,
    book_name   = BOOK_NAME,
    n           = 5,                 # 생성할 문제 수
    quiz_type   = "multiple_choice", # multiple_choice / ox / short_answer / fill_in_blank
    source      = "random",          # summary / quiz / random
    difficulty  = "medium",          # easy / medium / hard
    save        = True,
    print_result= True,
)

print(f"\n✅ 생성된 퀴즈: {result['meta']['n']}개")

In [ ]:
# 여러 타입 연속 생성 예시
for qtype in ["ox", "short_answer"]:
    r = generate_quiz(cfg, BOOK_NAME, n=3, quiz_type=qtype,
                      difficulty="medium", print_result=False, save=True)
    print(f"✅ {qtype}: {r['meta']['n']}개 생성")

---
## 📐 5단계: Graph RAG

수식·변수·개념 간 관계 그래프를 구축하고 Obsidian Graph View용 `.md`로 내보냅니다.

In [ ]:
from study_assistant import run_graph_rag

graph = run_graph_rag(
    cfg,
    book_name       = BOOK_NAME,
    visualize       = True,   # pyvis HTML 시각화 생성
    obsidian_export = True,   # Obsidian용 .md 파일 생성
)

ents = graph["entities"]
print(f"\n📊 그래프 요약")
print(f"  수식:  {len(ents['formulas'])}개")
print(f"  변수:  {len(ents['variables'])}개")
print(f"  개념:  {len(ents['concepts'])}개")
print(f"  엣지:  {len(graph['edges'])}개")

In [ ]:
from study_assistant import load_graph, query_formulas

# 저장된 그래프 로드
graph = load_graph(cfg, BOOK_NAME)

if graph:
    # 키워드로 수식/개념 검색
    query = input("검색어 입력 (예: 분산, 기대값, sigma): ").strip()
    result = query_formulas(graph, query)
    print("\n" + result)
else:
    print("graph RAG를 먼저 실행해주세요.")

In [ ]:
# pyvis HTML 시각화 Colab에서 열기
from pathlib import Path
from IPython.display import IFrame

html_path = cfg.book_vault_dir(BOOK_NAME) / "formula_graph.html"
if html_path.exists():
    display(IFrame(str(html_path), width="100%", height="600"))
else:
    print("formula_graph.html 없음 — visualize=True로 run_graph_rag를 실행하세요.")

---
## 🛠️ 유틸리티

In [ ]:
# 전체 데이터 현황 한눈에 보기
from study_assistant import print_all
print_all(cfg)

In [ ]:
# vault_index.json 통계
import json
from pathlib import Path

idx_path = cfg.book_vault_dir(BOOK_NAME) / "vault_index.json"
if idx_path.exists():
    idx = json.loads(idx_path.read_text(encoding="utf-8"))
    print(f"생성 시각:  {idx['generated_at']}")
    print(f"총 파일:   {idx['total_files']}개")
    print(f"총 개념:   {idx['stats']['total_concepts']}개")
    print(f"총 청크:   {idx['stats']['total_chunks']}개")
    print("\n파일 타입별:")
    for t, n in sorted(idx["stats"]["by_type"].items()):
        print(f"  {t}: {n}개")
else:
    print("파이프라인을 먼저 실행해주세요.")

In [ ]:
# 설정 고급 조작 예시
cfg2 = Config()

# 특정 값만 오버라이드
cfg2.set("pipeline.max_concurrent", 3)   # 동시 처리 수 조절
cfg2.set("pipeline.max_retries", 5)       # 재시도 횟수
cfg2.set("chunking.mode", "manual")       # 수동 청킹 모드
cfg2.set("book.index", 1)                 # 두 번째 교재 선택

# yaml 파일로부터 로드 (오버라이드)
# cfg2.load("/content/drive/MyDrive/my_config.yaml")

# dict로부터 로드
cfg2.load({"ocr": {"batch_size": 30, "max_concurrent": 2}})

cfg2.show()

# 현재 설정 저장
# cfg2.save("/content/drive/MyDrive/config_backup.yaml")

---
## 🚀 전체 파이프라인 한 번에 실행

In [ ]:
import os
from study_assistant import Config, run_ocr, run_chunking, run_pipeline, list_pdfs

# ── 설정 ────────────────────────────────────────────────
cfg = Config()
cfg.ensure_dirs()

pdfs = list_pdfs(cfg.pdf_dir)
if not pdfs:
    raise FileNotFoundError(f"{cfg.pdf_dir} 에 PDF 파일을 넣어주세요.")

selected  = pdfs[cfg.book_index]
book_name = selected["name"]
pdf_path  = str(selected["path"])

print(f"🎯 대상 교재: {book_name}\n")

# ── 1. OCR ──────────────────────────────────────────────
ocr_dir = cfg.book_ocr_dir(book_name)
if not any(ocr_dir.glob("page_*.md")):
    print("[1/3] OCR 실행...")
    run_ocr(cfg, pdf_path, book_name)
else:
    print(f"[1/3] OCR 캐시 사용: {len(list(ocr_dir.glob('page_*.md')))}p")

# ── 2. 청킹 ─────────────────────────────────────────────
print("[2/3] 청킹 실행 (자동 모드)...")
chunks = run_chunking(cfg, book_name, interactive=False)
print(f"       청크 {len(chunks)}개 완료")

# ── 3. 파이프라인 ───────────────────────────────────────
print("[3/3] 파이프라인 실행...")
results, vault_dir = run_pipeline(cfg, book_name, overwrite=True)

print(f"\n🎉 완료!")
print(f"   Vault: {vault_dir}")
print(f"   파일:  {len(results)}개")